# Measure policy lookup paths honestly

**Evidence state:** Measured

## What this demonstrates

Measure four different runtime paths: precompiled dictionary action lookup, dictionary values plus argmax, VPM action-only lookup, and VPM full-trace lookup.

## Why it matters

Benchmarks become misleading when different outputs are treated as equivalent. This demonstration keeps the action-only and trace-rich paths separate and records the environment that produced the numbers.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")

## Source and package mapping

- `examples/policy_lookup_benchmark.py`
- `examples/arcade_shooter_policy.py`
- `zeromodel`
- `zeromodel-video`


In [ ]:
import json
import matplotlib.pyplot as plt
from examples.policy_lookup_benchmark import run_benchmark

result = run_benchmark(lookups=50_000, repeat=3)
print(json.dumps(result, indent=2, sort_keys=True))

In [ ]:
labels = list(result["nanoseconds_per_lookup"])
values = [result["nanoseconds_per_lookup"][label] for label in labels]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(labels, values)
ax.set_xlabel("best nanoseconds per lookup on this runner")
ax.set_title("Different outputs have different costs")
plt.show()

## Application

```text
runtime requirement -> choose required output contract -> benchmark that exact path
```

This allows an application to decide whether it needs only an action, source values plus argmax, or a full evidence trace.

## Boundaries and limitations

These numbers describe one GitHub runner or local machine. They are not universal performance claims. Interpreter version, processor, load, and package versions affect the result.

## Reproduction record

`execution.json` records the Git revision, Python and platform details, runner architecture, relevant package versions, notebook command, and raw benchmark output.


In [ ]:
print(
    json.dumps(
        {
            "demo_id": "policy-lookup-benchmark",
            "artifact_id": result["artifact_id"],
            "lookups": result["lookups"],
            "repeat": result["repeat"],
            "note": result["note"],
        },
        indent=2,
    )
)